In [9]:
# -*- coding: utf-8 -*-
"""
Guardiões da IA — Central de Automação PREMIUM EDITION
Visual moderno e profissional com animações e design sofisticado

Autor: Tiago Felix
Data: 2026-01-23
"""

import os
import sys
import json
import queue
import threading
import subprocess
import shlex
import time
import tempfile
from pathlib import Path
from datetime import datetime
import platform
import tkinter as tk
from tkinter import ttk, messagebox
import webbrowser
import shutil
import nbformat

# =========================
# CONFIGURAÇÃO BÁSICA
# =========================

BASE_DIR = r"C:\Users\126815"

ITEMS = [
    {"label": "Gestor NFS ULTRA", "path": "RPA_GESTOR_NFS_ULTRA.ipynb", "kind": "ipynb", "icon": "📊", "color": "#3B82F6"},
    {"label": "Leitor Lumicenter", "path": "RPA_LEITOR_LUMICEN.ipynb", "kind": "ipynb", "icon": "💡", "color": "#F59E0B"},
    {"label": "Leitor Persun", "path": "RPA_LEITOR_PERSUN.ipynb", "kind": "ipynb", "icon": "📖", "color": "#8B5CF6"},
    {"label": "N1 Zendesk", "path": "RPA_N1_ZENDESK.ipynb", "kind": "ipynb", "icon": "🎫", "color": "#10B981"},
    {"label": "Pedidos Coord (E-mail)", "path": "RPA_PEDIDOS_Coord_E-MAIL-.ipynb", "kind": "ipynb", "icon": "📧", "color": "#EF4444"},
    {"label": "RC SAP", "path": "RPA_RC_SAP.ipynb", "kind": "ipynb", "icon": "🔐", "color": "#06B6D4"},
    {"label": "Renomear HTB", "path": "RPA_RENOMEAR_HTB.ipynb", "kind": "ipynb", "icon": "📝", "color": "#F97316"},
    {"label": "Tickets (E-mail)", "path": "RPA_TICKETS_E-MAIL.ipynb", "kind": "ipynb", "icon": "🎟️", "color": "#EC4899"},
]

# Power BI - página "Menu"
PBI_MENU_URL = r"https://app.powerbi.com/groups/603030a4-7459-45f3-ad62-07560039fba9/reports/eb396a94-1c03-4751-b1e0-218f373e6ba1/e4668548a078e9dc9d0e?experience=power-bi"

# SharePoint Engenharia - link fornecido
SHAREPOINT_ENG_URL = (
    r"https://paguemenos-my.sharepoint.com/shared?"
    r"id=%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
    r"&listurl=https%3A%2F%2Fpaguemenos%2Esharepoint%2Ecom%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
)

# =========================
# Temas Premium (Dark + Light)
# =========================

DARK_THEME = {
    "BG": "#0A0E1A",
    "BG_LIGHT": "#131827",
    "SURFACE": "#1E293B",
    "BORDER": "#334155",
    "PRIMARY": "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT": "#10B981",
    "ERROR": "#EF4444",
    "TEXT": "#F1F5F9",
    "TEXT_MUTED": "#94A3B8",
    "GLOW": "#60A5FA",
}

LIGHT_THEME = {
    "BG": "#F8FAFC",
    "BG_LIGHT": "#FFFFFF",
    "SURFACE": "#FFFFFF",
    "BORDER": "#E2E8F0",
    "PRIMARY": "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT": "#10B981",
    "ERROR": "#EF4444",
    "TEXT": "#0F172A",
    "TEXT_MUTED": "#64748B",
    "GLOW": "#3B82F6",
}

def apply_premium_theme(root: tk.Tk, theme_dict: dict):
    style = ttk.Style(root)
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass

    root.configure(bg=theme_dict["BG"])
    style.configure(".", background=theme_dict["BG"], foreground=theme_dict["TEXT"], font=("Segoe UI", 11))
    style.configure("TFrame", background=theme_dict["BG"])
    return theme_dict

# =========================
# Logger Premium
# =========================

class PremiumLogger:
    def __init__(self, text_widget: tk.Text, max_lines: int = 20000):
        self.text = text_widget
        self.max_lines = max_lines
        self.text.configure(state="disabled")
        self._setup_tags()

    def _setup_tags(self):
        self.text.tag_config("success", foreground="#10B981")
        self.text.tag_config("error", foreground="#EF4444")
        self.text.tag_config("warning", foreground="#F59E0B")
        self.text.tag_config("info", foreground="#3B82F6")

    def write(self, msg: str):
        try:
            self.text.configure(state="normal")

            tag = None
            low = msg.lower()
            if "sucesso" in low or "✅" in msg:
                tag = "success"
            elif "erro" in low or "❌" in msg:
                tag = "error"
            elif "aviso" in low or "⚠️" in msg:
                tag = "warning"
            elif "[run]" in msg or "[kernels]" in msg or "🚀" in msg:
                tag = "info"

            if tag:
                self.text.insert("end", msg, tag)
            else:
                self.text.insert("end", msg)

            if int(self.text.index('end-1c').split('.')[0]) > self.max_lines:
                self.text.delete("1.0", "2.0")

            self.text.see("end")
            self.text.configure(state="disabled")
            self.text.update_idletasks()  # ✅ Força atualização visual imediata
        except Exception as e:
            # Silenciosamente ignora erros de encoding no logger
            pass

    def flush(self):
        try:
            self.text.update_idletasks()
        except:
            pass

# =========================
# Helpers
# =========================

def _open_default(url: str):
    try:
        webbrowser.open(url, new=2)
        print(f"[dashboard] Abrindo: {url}\n")
    except Exception as e:
        print(f"[dashboard] Falha ao abrir URL: {e}\n")

def open_dash_menu():
    _open_default(PBI_MENU_URL)

def open_sharepoint_engenharia():
    _open_default(SHAREPOINT_ENG_URL)

def pick_kernel_name(prefer="python3"):
    try:
        proc = subprocess.run(
            ["jupyter", "kernelspec", "list", "--json"],
            capture_output=True, text=True, check=True
        )
    except Exception as e:
        print(f"[kernels] Erro ao listar kernels: {e}")
        return None

    try:
        data = json.loads(proc.stdout)
        kernels = list(data.get("kernelspecs", {}).keys())
        print(f"[kernels] Kernels disponíveis: {kernels}")
        if prefer in kernels:
            return prefer
        return kernels[0] if kernels else None
    except Exception as e:
        print(f"[kernels] Erro ao processar kernels: {e}")
        return None

def _terminate_proc_tree(proc: subprocess.Popen):
    if proc is None:
        return
    try:
        if platform.system() == "Windows":
            subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                           capture_output=True, check=True)
        else:
            proc.terminate()
    except:
        pass

def _popen_stream(cmd: str, env=None, cwd=None, on_start=None):
    print(f"[run] Executando comando: {cmd}")
    
    # Garante encoding UTF-8 no Windows
    if platform.system() == "Windows":
        creation_flags = subprocess.CREATE_NO_WINDOW
    else:
        creation_flags = 0
    
    proc = subprocess.Popen(
        cmd, 
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, 
        text=True,
        encoding='utf-8',  # ✅ Força UTF-8
        errors='replace',  # ✅ Substitui caracteres inválidos
        env=env, 
        cwd=cwd,
        creationflags=creation_flags
    )

    if on_start:
        on_start(proc)

    while True:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        print(line, end="")
        sys.stdout.flush()  # ✅ Força flush imediato
    
    return proc.wait()

def run_notebook_as_script(path_ipynb: str, on_start=None):
    path_ipynb = os.path.abspath(path_ipynb)
    if not os.path.exists(path_ipynb):
        raise FileNotFoundError(f"Notebook não encontrado: {path_ipynb}")

    nb_dir = os.path.dirname(path_ipynb)
    nb_name = os.path.splitext(os.path.basename(path_ipynb))[0]

    with tempfile.TemporaryDirectory() as tmpdir:
        cmd_convert = ['jupyter', 'nbconvert', '--to', 'script', path_ipynb, '--output-dir', tmpdir]
        code_conv = _popen_stream(cmd_convert, cwd=nb_dir, on_start=on_start)

        if code_conv != 0:
            return code_conv

        out_py = os.path.join(tmpdir, nb_name + ".py")

        if not os.path.exists(out_py):
            cand = list(Path(tmpdir).glob("*.py"))
            if not cand:
                return 1
            out_py = str(cand[0])

        cmd_run = [sys.executable, out_py]
        return _popen_stream(cmd_run, cwd=nb_dir, on_start=on_start)

def run_notebook(path_ipynb: str, on_start=None):
    path_ipynb = os.path.abspath(path_ipynb)
    nb_dir = os.path.dirname(path_ipynb)
    out_base = os.path.splitext(os.path.basename(path_ipynb))[0] + "_executado"

    kernel = pick_kernel_name()
    if not kernel:
        print("[run] Nenhum kernel Jupyter encontrado, usando execução como script...")
        return run_notebook_as_script(path_ipynb, on_start)

    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"  # ✅ Força encoding UTF-8

    cmd1 = [
        'jupyter', 'nbconvert', '--to', 'notebook', '--execute', path_ipynb,
        '--output', out_base, '--ExecutePreprocessor.kernel_name', kernel,
        '--ExecutePreprocessor.timeout=0'
    ]
    print(f"[run] Tentando método 1 (nbconvert): {' '.join(cmd1)}")
    
    code1 = _popen_stream(cmd1, env=env, cwd=nb_dir, on_start=on_start)
    if code1 == 0:
        return 0

    print("[run] Método 1 falhou, tentando método 2 (jupyter run)...")
    cmd2 = ['jupyter', 'run', '--kernel', kernel, path_ipynb]
    code2 = _popen_stream(cmd2, env=env, cwd=nb_dir, on_start=on_start)
    if code2 == 0:
        return 0

    print("[run] Método 2 falhou, tentando execução como script...")
    return run_notebook_as_script(path_ipynb, on_start=on_start)

def run_python(path_py: str, on_start=None):
    path_py = os.path.abspath(path_py)
    py_dir = os.path.dirname(path_py)
    cmd = [sys.executable, path_py]
    return _popen_stream(cmd, cwd=py_dir, on_start=on_start)

# =========================
# App Premium
# =========================

class GuardioesIAPremium(tk.Tk):
    def __init__(self, base_dir_scripts: str, items: list[dict]):
        super().__init__()
        self.title("🛡️ Guardiões da IA - Premium Edition")
        self.geometry("1450x900")
        self.minsize(1300, 780)

        self.is_dark_mode = True
        self.colors = apply_premium_theme(self, DARK_THEME)

        # 🔥 Ajuste global de fontes
        self.base_font = ("Segoe UI", 12)
        self.small_font = ("Segoe UI", 11)
        self.medium_font = ("Segoe UI Semibold", 14)
        self.big_font = ("Segoe UI Semibold", 18)
        self.icon_font = ("Segoe UI Emoji", 32)
        self.terminal_font = ("Cascadia Code", 12)

        self.current_worker = None
        self.current_proc = None
        self.running_task_name = None

        self.base_dir = base_dir_scripts
        self.items = items

        self._build_header()
        self._build_main_content()

        self._logger = PremiumLogger(self.txt_log)
        sys.stdout = self._logger
        sys.stderr = self._logger

        self.bind_all("<Control-m>", lambda e: open_dash_menu())
        self.bind_all("<Control-e>", lambda e: open_sharepoint_engenharia())
        self.bind_all("<Control-t>", lambda e: self._toggle_theme())

        self._tick_clock()
        self._animate_progress()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ------------ HEADER ------------

    def _build_header(self):
        header = tk.Frame(self, bg=self.colors["BG"], height=140)
        header.pack(fill="x")
        header.pack_propagate(False)

        self.header_glass = tk.Frame(
            header,
            bg=self.colors["BG_LIGHT"],
            highlightbackground=self.colors["BORDER"],
            highlightthickness=1
        )
        self.header_glass.pack(fill="both", expand=True, padx=25, pady=20)

        # ESQUERDA
        left = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        left.pack(side="left", padx=20, pady=15)

        self.title_lbl = tk.Label(
            left, text="🛡️ Guardiões da IA",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 20, "bold")
        )
        self.title_lbl.pack(anchor="w")

        self.subtitle_lbl = tk.Label(
            left, text="Central de Automação Premium | Escolha sua missão",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 13)
        )
        self.subtitle_lbl.pack(anchor="w", pady=(6, 0))

        # DESENVOLVEDOR CENTRALIZADO NO MEIO
        center = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        center.pack(side="left", expand=True, fill="both")

        dev_center = tk.Frame(center, bg=self.colors["BG_LIGHT"])
        dev_center.place(relx=0.5, rely=0.5, anchor="center")

        tk.Label(
            dev_center, text="🚀 Desenvolvido por:",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 11, "bold")
        ).pack()

        tk.Label(
            dev_center, text="TIAGO FELIX",
            bg=self.colors["BG_LIGHT"], fg=self.colors["GLOW"],
            font=("Segoe UI Semibold", 16, "bold")
        ).pack(pady=(2, 0))

        # DIREITA
        right = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        right.pack(side="right", padx=15, pady=15)

        buttons = tk.Frame(right, bg=self.colors["BG_LIGHT"])
        buttons.pack(anchor="e", pady=(0, 10))

        # Tema
        self.btn_theme = tk.Button(
            buttons, text="☀️ Modo Claro", command=self._toggle_theme,
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            relief="flat", font=self.small_font, padx=14, pady=7,
            cursor="hand2"
        )
        self.btn_theme.pack(side="left", padx=(0, 10))

        # Power BI (somente link)
        self.btn_pbi = tk.Button(
            buttons, text="📊 Power BI (Navegador)", command=open_dash_menu,
            bg=self.colors["PRIMARY"], fg="white", relief="flat",
            font=self.medium_font, padx=18, pady=8,
            cursor="hand2"
        )
        self.btn_pbi.pack(side="left", padx=(0, 10))

        # SharePoint Engenharia
        self.btn_sp = tk.Button(
            buttons, text="🏗️ Engenharia (SharePoint)", command=open_sharepoint_engenharia,
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"], relief="flat",
            font=self.small_font, padx=14, pady=7,
            cursor="hand2"
        )
        self.btn_sp.pack(side="left")

        # Relógio no canto direito
        self.clock_var = tk.StringVar()
        self.clock_label = tk.Label(
            right, textvariable=self.clock_var,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 11)
        )
        self.clock_label.pack(anchor="e", pady=(10, 0))

    # ------------ MAIN ------------

    def _build_main_content(self):
        self.main = tk.Frame(self, bg=self.colors["BG"])
        self.main.pack(fill="both", expand=True, padx=25, pady=(10, 20))

        self.main.grid_columnconfigure(0, weight=2)
        self.main.grid_columnconfigure(1, weight=3)
        self.main.grid_rowconfigure(0, weight=1)

        self._build_cards_section(self.main)
        self._build_terminal_section(self.main)

    def _build_cards_section(self, parent):
        left = tk.Frame(parent, bg=self.colors["BG"])
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 20))

        header = tk.Frame(left, bg=self.colors["BG"])
        header.pack(fill="x", pady=(0, 15))

        lbl = tk.Label(
            header, text="⚙️ Automatizações Disponíveis",
            bg=self.colors["BG"], fg=self.colors["TEXT"], 
            font=("Segoe UI Semibold", 16)
        )
        lbl.pack(anchor="w")

        # Frame com scroll
        canvas_frame = tk.Frame(left, bg=self.colors["BG"])
        canvas_frame.pack(fill="both", expand=True)

        canvas = tk.Canvas(canvas_frame, bg=self.colors["BG"], highlightthickness=0)
        scroll = ttk.Scrollbar(canvas_frame, orient="vertical", command=canvas.yview)
        frame = tk.Frame(canvas, bg=self.colors["BG"])

        frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=frame, anchor="nw")
        canvas.configure(yscrollcommand=scroll.set)

        canvas.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        for item in self.items:
            self._create_card(frame, item)

    def _create_card(self, parent, item):
        card = tk.Frame(
            parent, bg=self.colors["SURFACE"],
            highlightbackground=self.colors["BORDER"], highlightthickness=1,
            cursor="hand2"
        )
        card.pack(fill="x", pady=12, ipady=14, ipadx=16)

        def on_enter(e):
            card.configure(bg=self.colors["BG_LIGHT"], highlightbackground=item["color"])
            for child in card.winfo_children():
                if isinstance(child, tk.Frame):
                    child.configure(bg=self.colors["BG_LIGHT"])
                    for subchild in child.winfo_children():
                        if isinstance(subchild, (tk.Frame, tk.Label)):
                            subchild.configure(bg=self.colors["BG_LIGHT"])
        def on_leave(e):
            card.configure(bg=self.colors["SURFACE"], highlightbackground=self.colors["BORDER"])
            for child in card.winfo_children():
                if isinstance(child, tk.Frame):
                    child.configure(bg=self.colors["SURFACE"])
                    for subchild in child.winfo_children():
                        if isinstance(subchild, (tk.Frame, tk.Label)):
                            subchild.configure(bg=self.colors["SURFACE"])

        card.bind("<Enter>", on_enter)
        card.bind("<Leave>", on_leave)
        card.bind("<Button-1>", lambda e: self._start_task(item))

        left = tk.Frame(card, bg=self.colors["SURFACE"])
        left.pack(side="left", fill="both", expand=True)
        left.bind("<Button-1>", lambda e: self._start_task(item))

        icon = tk.Label(
            left, text=item["icon"], bg=self.colors["SURFACE"],
            fg=item["color"], font=("Segoe UI Emoji", 36)
        )
        icon.pack(side="left", padx=(8, 18))
        icon.bind("<Button-1>", lambda e: self._start_task(item))

        text_frame = tk.Frame(left, bg=self.colors["SURFACE"])
        text_frame.pack(side="left", fill="both", expand=True)
        text_frame.bind("<Button-1>", lambda e: self._start_task(item))

        title = tk.Label(
            text_frame, text=item["label"],
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 14)
        )
        title.pack(anchor="w", pady=(2, 0))
        title.bind("<Button-1>", lambda e: self._start_task(item))

        desc = tk.Label(
            text_frame, text=f"Executar {item['kind'].upper()}",
            bg=self.colors["SURFACE"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 11)
        )
        desc.pack(anchor="w")
        desc.bind("<Button-1>", lambda e: self._start_task(item))

        arrow = tk.Label(
            card, text="▶", bg=self.colors["SURFACE"],
            fg=item["color"], font=("Segoe UI", 20)
        )
        arrow.pack(side="right", padx=14)
        arrow.bind("<Button-1>", lambda e: self._start_task(item))

    # ------------ TERMINAL ------------

    def _build_terminal_section(self, parent):
        right = tk.Frame(parent, bg=self.colors["BG"])
        right.grid(row=0, column=1, sticky="nsew")

        # Status
        status_card = tk.Frame(right, bg=self.colors["SURFACE"],
                               highlightbackground=self.colors["BORDER"],
                               highlightthickness=1)
        status_card.pack(fill="x", pady=(0, 18))

        progress_frame = tk.Frame(status_card, bg=self.colors["SURFACE"])
        progress_frame.pack(fill="x", padx=20, pady=(20, 12))

        self.progress_canvas = tk.Canvas(progress_frame, height=12,
                                         bg=self.colors["BG_LIGHT"],
                                         highlightthickness=0)
        self.progress_canvas.pack(fill="x", expand=True, side="left")
        self.progress_bar = self.progress_canvas.create_rectangle(0, 0, 0, 12,
                                                                  fill=self.colors["PRIMARY"],
                                                                  outline="")

        self.btn_stop = tk.Button(
            progress_frame, text="⏹ Parar", command=self._stop_task,
            bg=self.colors["ERROR"], fg="white", relief="flat",
            font=self.small_font, padx=20, pady=8,
            state="disabled", cursor="hand2"
        )
        self.btn_stop.pack(side="right", padx=(15, 0))

        self.status_var = tk.StringVar(value="🎯 Pronto para iniciar automação")
        self.status_label = tk.Label(
            status_card, textvariable=self.status_var,
            bg=self.colors["SURFACE"], fg=self.colors["PRIMARY"],
            font=("Segoe UI Semibold", 14), anchor="w"
        )
        self.status_label.pack(fill="x", padx=20, pady=(0, 20))

        # Terminal
        terminal_frame = tk.Frame(right, bg=self.colors["BG"])
        terminal_frame.pack(fill="both", expand=True)

        header = tk.Frame(terminal_frame, bg=self.colors["SURFACE"],
                          highlightbackground=self.colors["BORDER"],
                          highlightthickness=1)
        header.pack(fill="x")

        tk.Label(
            header, text="💻 Terminal de Execução",
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 15)
        ).pack(side="left", padx=20, pady=14)

        btns = tk.Frame(header, bg=self.colors["SURFACE"])
        btns.pack(side="right", padx=20, pady=14)

        tk.Button(
            btns, text="🗑️ Limpar", command=self._clear_log,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            relief="flat", font=self.small_font,
            padx=14, pady=6, cursor="hand2"
        ).pack(side="left", padx=(0, 12))

        tk.Button(
            btns, text="📋 Copiar", command=self._copy_log,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            relief="flat", font=self.small_font,
            padx=14, pady=6, cursor="hand2"
        ).pack(side="left")

        # Corpo do terminal
        terminal_body = tk.Frame(terminal_frame, bg=self.colors["BG_LIGHT"],
                                 highlightbackground=self.colors["BORDER"],
                                 highlightthickness=1)
        terminal_body.pack(fill="both", expand=True)

        terminal_bg = "#0D1117" if self.is_dark_mode else "#F6F8FA"
        terminal_fg = "#C9D1D9" if self.is_dark_mode else "#24292F"
        terminal_cursor = "#58A6FF" if self.is_dark_mode else "#0969DA"

        self.txt_log = tk.Text(
            terminal_body, wrap="word",
            bg=terminal_bg, fg=terminal_fg,
            insertbackground=terminal_cursor,
            relief="flat", padx=22, pady=20,
            font=("Cascadia Code", 13)
        )
        self.txt_log.pack(side="left", fill="both", expand=True)

        scrollbar = ttk.Scrollbar(terminal_body, orient="vertical",
                                  command=self.txt_log.yview)
        scrollbar.pack(side="right", fill="y")
        self.txt_log.configure(yscrollcommand=scrollbar.set)

    # ---------- Execução ----------

    def _animate_progress(self):
        if hasattr(self, "progress_value"):
            width = self.progress_canvas.winfo_width()
            self.progress_canvas.coords(
                self.progress_bar,
                0, 0,
                width * self.progress_value, 12
            )
        self.after(50, self._animate_progress)

    def _on_proc_start(self, proc):
        self.current_proc = proc
        self.btn_stop.configure(state="normal")

    def _start_task(self, item):
        if self.current_worker and self.current_worker.is_alive():
            messagebox.showinfo(
                "⚠️ Execução em andamento",
                f"Já existe uma automação executando:\n{self.running_task_name}\n\nAguarde terminar."
            )
            return

        self.running_task_name = item["label"]
        self.progress_value = 0.0
        self._set_status(f"🚀 Executando: {item['label']}", "info")
        self._disable_all(True)
        self.btn_stop.configure(state="disabled")
        self.current_proc = None

        self.current_worker = threading.Thread(
            target=self._worker_run, args=(item,), daemon=True
        )
        self.current_worker.start()

    def _worker_run(self, item):
        try:
            for v in (0.05, 0.18, 0.35):
                self.progress_value = v
                time.sleep(0.12)

            kind = item["kind"]
            path = os.path.join(self.base_dir, item["path"])
            
            # Verifica se o arquivo existe
            if not os.path.exists(path):
                error_msg = f"❌ ARQUIVO NÃO ENCONTRADO: {path}"
                print(f"\n{'='*70}")
                print(error_msg)
                print(f"{'='*70}\n")
                self._set_status(error_msg, "error")
                return

            print("\n" + "="*70)
            print(f"🚀 Iniciando: {item['label']}")
            print(f"📂 Caminho: {path}")
            print("="*70 + "\n")

            if kind == "ipynb":
                code = run_notebook(path, on_start=self._on_proc_start)
            else:
                code = run_python(path, on_start=self._on_proc_start)

            self.progress_value = 1

            if code == 0:
                self._set_status(f"✅ {item['label']} concluído com sucesso!", "success")
            else:
                self._set_status(f"❌ {item['label']} terminou com código {code}", "error")

            print("\n" + "="*70)
            print(f"✅ Fim: {item['label']} (exitcode={code})")
            print("="*70 + "\n")

        except FileNotFoundError as e:
            error_msg = f"❌ Arquivo não encontrado:\n{e}"
            self._set_status(error_msg, "error")
            print(f"\n{error_msg}\n")

        except Exception as e:
            error_msg = f"❌ Erro inesperado:\n{e}"
            self._set_status(error_msg, "error")
            print(f"\n{error_msg}\n")

        finally:
            self._disable_all(False)
            self.btn_stop.configure(state="disabled")
            self.current_proc = None
            self.current_worker = None
            self.running_task_name = None

    def _stop_task(self):
        if not self.current_worker:
            self._set_status("ℹ️ Nenhuma automação em execução.", "info")
            return

        if self.current_proc is None:
            self._set_status("ℹ️ Processo ainda iniciando...", "info")
            return

        self._set_status("🛑 Encerrando automação...", "error")
        _terminate_proc_tree(self.current_proc)
        print("\n[STOP] Processo encerrado manualmente.\n")

    def _disable_all(self, disable=True):
        state = "disabled" if disable else "normal"

        preserve = {
            self.btn_stop, self.btn_pbi, self.btn_theme, self.btn_sp
        }

        def apply(widget):
            if isinstance(widget, (tk.Button, ttk.Button)):
                if widget not in preserve:
                    try:
                        widget.configure(state=state)
                    except:
                        pass
            for c in widget.winfo_children():
                apply(c)

        apply(self)

    def _set_status(self, text, kind):
        colors = {
            "success": self.colors["ACCENT"],
            "error": self.colors["ERROR"],
            "info": self.colors["PRIMARY"],
            "warning": "#F59E0B"
        }
        self.status_label.configure(fg=colors.get(kind, self.colors["TEXT"]))
        self.status_var.set(text)

    def _clear_log(self):
        self.txt_log.configure(state="normal")
        self.txt_log.delete("1.0", "end")
        self.txt_log.configure(state="disabled")

    def _copy_log(self):
        txt = self.txt_log.get("1.0", "end-1c")
        self.clipboard_clear()
        self.clipboard_append(txt)
        messagebox.showinfo("📋 Copiado", "Log copiado para a área de transferência!")

    def _tick_clock(self):
        self.clock_var.set(datetime.now().strftime("%d/%m/%Y %H:%M:%S"))
        self.after(1000, self._tick_clock)

    def _toggle_theme(self):
        self.is_dark_mode = not self.is_dark_mode
        self.colors = apply_premium_theme(self, DARK_THEME if self.is_dark_mode else LIGHT_THEME)
        
        # Atualiza texto do botão de tema
        self.btn_theme.configure(
            text="🌙 Modo Escuro" if not self.is_dark_mode else "☀️ Modo Claro"
        )
        
        # Reconstroi a interface com novo tema
        for widget in self.winfo_children():
            widget.destroy()
        
        self._build_header()
        self._build_main_content()
        
        # Reconecta o logger
        self._logger = PremiumLogger(self.txt_log)
        sys.stdout = self._logger
        sys.stderr = self._logger

    def _on_close(self):
        if self.current_worker and self.current_worker.is_alive():
            if not messagebox.askyesno("⚠️ Fechar", "Uma automação está rodando. Deseja encerrar?"):
                return
            self._stop_task()
            self.after(600, self.destroy)
        else:
            self.destroy()

# =========================
# Main
# =========================

if __name__ == "__main__":
    # Verifica se o diretório base existe
    if not os.path.exists(BASE_DIR):
        print(f"❌ Diretório base não encontrado: {BASE_DIR}")
        print(f"⚠️  Verifique se o caminho está correto.")
        print(f"📂 O diretório atual é: {os.getcwd()}")
        
        # Tenta usar o diretório atual como alternativa
        choice = input("Deseja usar o diretório atual como base? (s/n): ")
        if choice.lower() == 's':
            BASE_DIR = os.getcwd()
            print(f"✅ Usando diretório atual: {BASE_DIR}")
        else:
            input("Pressione Enter para sair...")
            sys.exit(1)
    
    app = GuardioesIAPremium(BASE_DIR, ITEMS)
    app.mainloop()

In [2]:
# -*- coding: utf-8 -*-
"""
Guardiões da IA — Central de Automação PREMIUM EDITION
Visual moderno e profissional com animações e design sofisticado
VERSÃO AJUSTADA PARA NOTEBOOKS

Autor: Tiago Felix
Data: 2026-01-23
Ajustado: 2026-01-27
"""

import os
import sys
import json
import queue
import threading
import subprocess
import shlex
import time
import tempfile
from pathlib import Path
from datetime import datetime
import platform
import tkinter as tk
from tkinter import ttk, messagebox
import webbrowser
import shutil
import nbformat

# =========================
# CONFIGURAÇÃO BÁSICA
# =========================

BASE_DIR = r"C:\Users\126815"

ITEMS = [
    {"label": "Gestor NFS ULTRA", "path": "RPA_GESTOR_NFS_ULTRA.ipynb", "kind": "ipynb", "icon": "📊", "color": "#3B82F6"},
    {"label": "Leitor Lumicenter", "path": "RPA_LEITOR_LUMICEN.ipynb", "kind": "ipynb", "icon": "💡", "color": "#F59E0B"},
    {"label": "Leitor Persun", "path": "RPA_LEITOR_PERSUN.ipynb", "kind": "ipynb", "icon": "📖", "color": "#8B5CF6"},
    {"label": "N1 Zendesk", "path": "RPA_N1_ZENDESK.ipynb", "kind": "ipynb", "icon": "🎫", "color": "#10B981"},
    {"label": "Pedidos Coord (E-mail)", "path": "RPA_PEDIDOS_Coord_E-MAIL-.ipynb", "kind": "ipynb", "icon": "📧", "color": "#EF4444"},
    {"label": "RC SAP", "path": "RPA_RC_SAP.ipynb", "kind": "ipynb", "icon": "🔐", "color": "#06B6D4"},
    {"label": "Renomear HTB", "path": "RPA_RENOMEAR_HTB.ipynb", "kind": "ipynb", "icon": "📝", "color": "#F97316"},
    {"label": "Tickets (E-mail)", "path": "RPA_TICKETS_E-MAIL.ipynb", "kind": "ipynb", "icon": "🎟️", "color": "#EC4899"},
]

# Power BI - página "Menu"
PBI_MENU_URL = r"https://app.powerbi.com/groups/603030a4-7459-45f3-ad62-07560039fba9/reports/eb396a94-1c03-4751-b1e0-218f373e6ba1/e4668548a078e9dc9d0e?experience=power-bi"

# SharePoint Engenharia - link fornecido
SHAREPOINT_ENG_URL = (
    r"https://paguemenos-my.sharepoint.com/shared?"
    r"id=%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
    r"&listurl=https%3A%2F%2Fpaguemenos%2Esharepoint%2Ecom%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
)

# =========================
# Temas Premium (Dark + Light)
# =========================

DARK_THEME = {
    "BG": "#0A0E1A",
    "BG_LIGHT": "#131827",
    "SURFACE": "#1E293B",
    "BORDER": "#334155",
    "PRIMARY": "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT": "#10B981",
    "ERROR": "#EF4444",
    "TEXT": "#F1F5F9",
    "TEXT_MUTED": "#94A3B8",
    "GLOW": "#60A5FA",
}

LIGHT_THEME = {
    "BG": "#F8FAFC",
    "BG_LIGHT": "#FFFFFF",
    "SURFACE": "#FFFFFF",
    "BORDER": "#E2E8F0",
    "PRIMARY": "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT": "#10B981",
    "ERROR": "#EF4444",
    "TEXT": "#0F172A",
    "TEXT_MUTED": "#64748B",
    "GLOW": "#3B82F6",
}

def apply_premium_theme(root: tk.Tk, theme_dict: dict):
    style = ttk.Style(root)
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass

    root.configure(bg=theme_dict["BG"])
    style.configure(".", background=theme_dict["BG"], foreground=theme_dict["TEXT"], font=("Segoe UI", 10))
    style.configure("TFrame", background=theme_dict["BG"])
    return theme_dict

# =========================
# Logger Premium
# =========================

class PremiumLogger:
    def __init__(self, text_widget: tk.Text, max_lines: int = 20000):
        self.text = text_widget
        self.max_lines = max_lines
        self.text.configure(state="disabled")
        self._setup_tags()

    def _setup_tags(self):
        self.text.tag_config("success", foreground="#10B981")
        self.text.tag_config("error", foreground="#EF4444")
        self.text.tag_config("warning", foreground="#F59E0B")
        self.text.tag_config("info", foreground="#3B82F6")

    def write(self, msg: str):
        try:
            self.text.configure(state="normal")

            tag = None
            low = msg.lower()
            if "sucesso" in low or "✅" in msg:
                tag = "success"
            elif "erro" in low or "❌" in msg:
                tag = "error"
            elif "aviso" in low or "⚠️" in msg:
                tag = "warning"
            elif "[run]" in msg or "[kernels]" in msg or "🚀" in msg:
                tag = "info"

            if tag:
                self.text.insert("end", msg, tag)
            else:
                self.text.insert("end", msg)

            if int(self.text.index('end-1c').split('.')[0]) > self.max_lines:
                self.text.delete("1.0", "2.0")

            self.text.see("end")
            self.text.configure(state="disabled")
            self.text.update_idletasks()
        except Exception as e:
            pass

    def flush(self):
        try:
            self.text.update_idletasks()
        except:
            pass

# =========================
# Helpers
# =========================

def _open_default(url: str):
    try:
        webbrowser.open(url, new=2)
        print(f"[dashboard] Abrindo: {url}\n")
    except Exception as e:
        print(f"[dashboard] Falha ao abrir URL: {e}\n")

def open_dash_menu():
    _open_default(PBI_MENU_URL)

def open_sharepoint_engenharia():
    _open_default(SHAREPOINT_ENG_URL)

def pick_kernel_name(prefer="python3"):
    try:
        proc = subprocess.run(
            ["jupyter", "kernelspec", "list", "--json"],
            capture_output=True, text=True, check=True
        )
    except Exception as e:
        print(f"[kernels] Erro ao listar kernels: {e}")
        return None

    try:
        data = json.loads(proc.stdout)
        kernels = list(data.get("kernelspecs", {}).keys())
        print(f"[kernels] Kernels disponíveis: {kernels}")
        if prefer in kernels:
            return prefer
        return kernels[0] if kernels else None
    except Exception as e:
        print(f"[kernels] Erro ao processar kernels: {e}")
        return None

def _terminate_proc_tree(proc: subprocess.Popen):
    if proc is None:
        return
    try:
        if platform.system() == "Windows":
            subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                           capture_output=True, check=True)
        else:
            proc.terminate()
    except:
        pass

def _popen_stream(cmd: str, env=None, cwd=None, on_start=None):
    print(f"[run] Executando comando: {cmd}")
    
    if platform.system() == "Windows":
        creation_flags = subprocess.CREATE_NO_WINDOW
    else:
        creation_flags = 0
    
    proc = subprocess.Popen(
        cmd, 
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, 
        text=True,
        encoding='utf-8',
        errors='replace',
        env=env, 
        cwd=cwd,
        creationflags=creation_flags
    )

    if on_start:
        on_start(proc)

    while True:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        print(line, end="")
        sys.stdout.flush()
    
    return proc.wait()

def run_notebook_as_script(path_ipynb: str, on_start=None):
    path_ipynb = os.path.abspath(path_ipynb)
    if not os.path.exists(path_ipynb):
        raise FileNotFoundError(f"Notebook não encontrado: {path_ipynb}")

    nb_dir = os.path.dirname(path_ipynb)
    nb_name = os.path.splitext(os.path.basename(path_ipynb))[0]

    with tempfile.TemporaryDirectory() as tmpdir:
        cmd_convert = ['jupyter', 'nbconvert', '--to', 'script', path_ipynb, '--output-dir', tmpdir]
        code_conv = _popen_stream(cmd_convert, cwd=nb_dir, on_start=on_start)

        if code_conv != 0:
            return code_conv

        out_py = os.path.join(tmpdir, nb_name + ".py")

        if not os.path.exists(out_py):
            cand = list(Path(tmpdir).glob("*.py"))
            if not cand:
                return 1
            out_py = str(cand[0])

        cmd_run = [sys.executable, out_py]
        return _popen_stream(cmd_run, cwd=nb_dir, on_start=on_start)

def run_notebook(path_ipynb: str, on_start=None):
    path_ipynb = os.path.abspath(path_ipynb)
    nb_dir = os.path.dirname(path_ipynb)
    out_base = os.path.splitext(os.path.basename(path_ipynb))[0] + "_executado"

    kernel = pick_kernel_name()
    if not kernel:
        print("[run] Nenhum kernel Jupyter encontrado, usando execução como script...")
        return run_notebook_as_script(path_ipynb, on_start)

    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"

    cmd1 = [
        'jupyter', 'nbconvert', '--to', 'notebook', '--execute', path_ipynb,
        '--output', out_base, '--ExecutePreprocessor.kernel_name', kernel,
        '--ExecutePreprocessor.timeout=0'
    ]
    print(f"[run] Tentando método 1 (nbconvert): {' '.join(cmd1)}")
    
    code1 = _popen_stream(cmd1, env=env, cwd=nb_dir, on_start=on_start)
    if code1 == 0:
        return 0

    print("[run] Método 1 falhou, tentando método 2 (jupyter run)...")
    cmd2 = ['jupyter', 'run', '--kernel', kernel, path_ipynb]
    code2 = _popen_stream(cmd2, env=env, cwd=nb_dir, on_start=on_start)
    if code2 == 0:
        return 0

    print("[run] Método 2 falhou, tentando execução como script...")
    return run_notebook_as_script(path_ipynb, on_start=on_start)

def run_python(path_py: str, on_start=None):
    path_py = os.path.abspath(path_py)
    py_dir = os.path.dirname(path_py)
    cmd = [sys.executable, path_py]
    return _popen_stream(cmd, cwd=py_dir, on_start=on_start)

# =========================
# App Premium (AJUSTADO PARA NOTEBOOKS)
# =========================

class GuardioesIAPremium(tk.Tk):
    def __init__(self, base_dir_scripts: str, items: list[dict]):
        super().__init__()
        self.title("🛡️ Guardiões da IA - Premium Edition")
        
        # ✅ AJUSTE AUTOMÁTICO PARA TELA DO NOTEBOOK
        screen_width = self.winfo_screenwidth()
        screen_height = self.winfo_screenheight()
        
        # Define tamanho como 90% da tela (máximo 1400x850)
        window_width = min(1400, int(screen_width * 0.90))
        window_height = min(850, int(screen_height * 0.88))
        
        # Centraliza a janela
        x = (screen_width - window_width) // 2
        y = max(0, (screen_height - window_height) // 2 - 20)  # Sobe um pouco
        
        self.geometry(f"{window_width}x{window_height}+{x}+{y}")
        self.minsize(1100, 650)  # Tamanho mínimo reduzido

        self.is_dark_mode = True
        self.colors = apply_premium_theme(self, DARK_THEME)

        # 🔥 Fontes ajustadas para notebooks
        self.base_font = ("Segoe UI", 10)
        self.small_font = ("Segoe UI", 9)
        self.medium_font = ("Segoe UI Semibold", 12)
        self.big_font = ("Segoe UI Semibold", 15)
        self.icon_font = ("Segoe UI Emoji", 28)
        self.terminal_font = ("Cascadia Code", 10)

        self.current_worker = None
        self.current_proc = None
        self.running_task_name = None

        self.base_dir = base_dir_scripts
        self.items = items

        self._build_header()
        self._build_main_content()

        self._logger = PremiumLogger(self.txt_log)
        sys.stdout = self._logger
        sys.stderr = self._logger

        self.bind_all("<Control-m>", lambda e: open_dash_menu())
        self.bind_all("<Control-e>", lambda e: open_sharepoint_engenharia())
        self.bind_all("<Control-t>", lambda e: self._toggle_theme())

        self._tick_clock()
        self._animate_progress()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ------------ HEADER (COMPACTO) ------------

    def _build_header(self):
        header = tk.Frame(self, bg=self.colors["BG"], height=110)  # Reduzido de 140
        header.pack(fill="x")
        header.pack_propagate(False)

        self.header_glass = tk.Frame(
            header,
            bg=self.colors["BG_LIGHT"],
            highlightbackground=self.colors["BORDER"],
            highlightthickness=1
        )
        self.header_glass.pack(fill="both", expand=True, padx=20, pady=12)  # Padding reduzido

        # ESQUERDA
        left = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        left.pack(side="left", padx=15, pady=10)

        self.title_lbl = tk.Label(
            left, text="🛡️ Guardiões da IA",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 16, "bold")  # Reduzido de 20
        )
        self.title_lbl.pack(anchor="w")

        self.subtitle_lbl = tk.Label(
            left, text="Central de Automação Premium",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 10)  # Reduzido de 13
        )
        self.subtitle_lbl.pack(anchor="w", pady=(4, 0))

        # CENTRO (DESENVOLVEDOR)
        center = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        center.pack(side="left", expand=True, fill="both")

        dev_center = tk.Frame(center, bg=self.colors["BG_LIGHT"])
        dev_center.place(relx=0.5, rely=0.5, anchor="center")

        tk.Label(
            dev_center, text="🚀 Desenvolvido por:",
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 9, "bold")
        ).pack()

        tk.Label(
            dev_center, text="TIAGO FELIX",
            bg=self.colors["BG_LIGHT"], fg=self.colors["GLOW"],
            font=("Segoe UI Semibold", 13, "bold")
        ).pack(pady=(1, 0))

        # DIREITA
        right = tk.Frame(self.header_glass, bg=self.colors["BG_LIGHT"])
        right.pack(side="right", padx=12, pady=10)

        buttons = tk.Frame(right, bg=self.colors["BG_LIGHT"])
        buttons.pack(anchor="e", pady=(0, 6))

        # Tema
        self.btn_theme = tk.Button(
            buttons, text="☀️", command=self._toggle_theme,
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            relief="flat", font=("Segoe UI", 14), padx=8, pady=4,
            cursor="hand2"
        )
        self.btn_theme.pack(side="left", padx=(0, 6))

        # Power BI
        self.btn_pbi = tk.Button(
            buttons, text="📊 Power BI", command=open_dash_menu,
            bg=self.colors["PRIMARY"], fg="white", relief="flat",
            font=self.small_font, padx=12, pady=5,
            cursor="hand2"
        )
        self.btn_pbi.pack(side="left", padx=(0, 6))

        # SharePoint
        self.btn_sp = tk.Button(
            buttons, text="🏗️ SharePoint", command=open_sharepoint_engenharia,
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"], relief="flat",
            font=self.small_font, padx=12, pady=5,
            cursor="hand2"
        )
        self.btn_sp.pack(side="left")

        # Relógio
        self.clock_var = tk.StringVar()
        self.clock_label = tk.Label(
            right, textvariable=self.clock_var,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 9)
        )
        self.clock_label.pack(anchor="e", pady=(6, 0))

    # ------------ MAIN ------------

    def _build_main_content(self):
        self.main = tk.Frame(self, bg=self.colors["BG"])
        self.main.pack(fill="both", expand=True, padx=20, pady=(8, 15))

        self.main.grid_columnconfigure(0, weight=2)
        self.main.grid_columnconfigure(1, weight=3)
        self.main.grid_rowconfigure(0, weight=1)

        self._build_cards_section(self.main)
        self._build_terminal_section(self.main)

    def _build_cards_section(self, parent):
        left = tk.Frame(parent, bg=self.colors["BG"])
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 15))

        header = tk.Frame(left, bg=self.colors["BG"])
        header.pack(fill="x", pady=(0, 10))

        lbl = tk.Label(
            header, text="⚙️ Automatizações",
            bg=self.colors["BG"], fg=self.colors["TEXT"], 
            font=("Segoe UI Semibold", 13)
        )
        lbl.pack(anchor="w")

        # Frame com scroll
        canvas_frame = tk.Frame(left, bg=self.colors["BG"])
        canvas_frame.pack(fill="both", expand=True)

        canvas = tk.Canvas(canvas_frame, bg=self.colors["BG"], highlightthickness=0)
        scroll = ttk.Scrollbar(canvas_frame, orient="vertical", command=canvas.yview)
        frame = tk.Frame(canvas, bg=self.colors["BG"])

        frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=frame, anchor="nw")
        canvas.configure(yscrollcommand=scroll.set)

        canvas.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        for item in self.items:
            self._create_card(frame, item)

    def _create_card(self, parent, item):
        card = tk.Frame(
            parent, bg=self.colors["SURFACE"],
            highlightbackground=self.colors["BORDER"], highlightthickness=1,
            cursor="hand2"
        )
        card.pack(fill="x", pady=8, ipady=10, ipadx=12)  # Padding reduzido

        def on_enter(e):
            card.configure(bg=self.colors["BG_LIGHT"], highlightbackground=item["color"])
            for child in card.winfo_children():
                if isinstance(child, tk.Frame):
                    child.configure(bg=self.colors["BG_LIGHT"])
                    for subchild in child.winfo_children():
                        if isinstance(subchild, (tk.Frame, tk.Label)):
                            subchild.configure(bg=self.colors["BG_LIGHT"])
        def on_leave(e):
            card.configure(bg=self.colors["SURFACE"], highlightbackground=self.colors["BORDER"])
            for child in card.winfo_children():
                if isinstance(child, tk.Frame):
                    child.configure(bg=self.colors["SURFACE"])
                    for subchild in child.winfo_children():
                        if isinstance(subchild, (tk.Frame, tk.Label)):
                            subchild.configure(bg=self.colors["SURFACE"])

        card.bind("<Enter>", on_enter)
        card.bind("<Leave>", on_leave)
        card.bind("<Button-1>", lambda e: self._start_task(item))

        left = tk.Frame(card, bg=self.colors["SURFACE"])
        left.pack(side="left", fill="both", expand=True)
        left.bind("<Button-1>", lambda e: self._start_task(item))

        icon = tk.Label(
            left, text=item["icon"], bg=self.colors["SURFACE"],
            fg=item["color"], font=("Segoe UI Emoji", 28)  # Reduzido de 36
        )
        icon.pack(side="left", padx=(6, 12))
        icon.bind("<Button-1>", lambda e: self._start_task(item))

        text_frame = tk.Frame(left, bg=self.colors["SURFACE"])
        text_frame.pack(side="left", fill="both", expand=True)
        text_frame.bind("<Button-1>", lambda e: self._start_task(item))

        title = tk.Label(
            text_frame, text=item["label"],
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 11)  # Reduzido de 14
        )
        title.pack(anchor="w", pady=(1, 0))
        title.bind("<Button-1>", lambda e: self._start_task(item))

        desc = tk.Label(
            text_frame, text=f"Executar {item['kind'].upper()}",
            bg=self.colors["SURFACE"], fg=self.colors["TEXT_MUTED"],
            font=("Segoe UI", 9)  # Reduzido de 11
        )
        desc.pack(anchor="w")
        desc.bind("<Button-1>", lambda e: self._start_task(item))

        arrow = tk.Label(
            card, text="▶", bg=self.colors["SURFACE"],
            fg=item["color"], font=("Segoe UI", 16)  # Reduzido de 20
        )
        arrow.pack(side="right", padx=10)
        arrow.bind("<Button-1>", lambda e: self._start_task(item))

    # ------------ TERMINAL ------------

    def _build_terminal_section(self, parent):
        right = tk.Frame(parent, bg=self.colors["BG"])
        right.grid(row=0, column=1, sticky="nsew")

        # Status (mais compacto)
        status_card = tk.Frame(right, bg=self.colors["SURFACE"],
                               highlightbackground=self.colors["BORDER"],
                               highlightthickness=1)
        status_card.pack(fill="x", pady=(0, 12))

        progress_frame = tk.Frame(status_card, bg=self.colors["SURFACE"])
        progress_frame.pack(fill="x", padx=15, pady=(12, 8))

        self.progress_canvas = tk.Canvas(progress_frame, height=10,
                                         bg=self.colors["BG_LIGHT"],
                                         highlightthickness=0)
        self.progress_canvas.pack(fill="x", expand=True, side="left")
        self.progress_bar = self.progress_canvas.create_rectangle(0, 0, 0, 10,
                                                                  fill=self.colors["PRIMARY"],
                                                                  outline="")

        self.btn_stop = tk.Button(
            progress_frame, text="⏹", command=self._stop_task,
            bg=self.colors["ERROR"], fg="white", relief="flat",
            font=("Segoe UI", 12), padx=12, pady=5,
            state="disabled", cursor="hand2"
        )
        self.btn_stop.pack(side="right", padx=(10, 0))

        self.status_var = tk.StringVar(value="🎯 Pronto para iniciar")
        self.status_label = tk.Label(
            status_card, textvariable=self.status_var,
            bg=self.colors["SURFACE"], fg=self.colors["PRIMARY"],
            font=("Segoe UI Semibold", 11), anchor="w"
        )
        self.status_label.pack(fill="x", padx=15, pady=(0, 12))

        # Terminal
        terminal_frame = tk.Frame(right, bg=self.colors["BG"])
        terminal_frame.pack(fill="both", expand=True)

        header = tk.Frame(terminal_frame, bg=self.colors["SURFACE"],
                          highlightbackground=self.colors["BORDER"],
                          highlightthickness=1)
        header.pack(fill="x")

        tk.Label(
            header, text="💻 Terminal",
            bg=self.colors["SURFACE"], fg=self.colors["TEXT"],
            font=("Segoe UI Semibold", 12)
        ).pack(side="left", padx=15, pady=10)

        btns = tk.Frame(header, bg=self.colors["SURFACE"])
        btns.pack(side="right", padx=15, pady=10)

        tk.Button(
            btns, text="🗑️", command=self._clear_log,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            relief="flat", font=("Segoe UI", 11),
            padx=10, pady=4, cursor="hand2"
        ).pack(side="left", padx=(0, 8))

        tk.Button(
            btns, text="📋", command=self._copy_log,
            bg=self.colors["BG_LIGHT"], fg=self.colors["TEXT"],
            relief="flat", font=("Segoe UI", 11),
            padx=10, pady=4, cursor="hand2"
        ).pack(side="left")

        # Corpo do terminal
        terminal_body = tk.Frame(terminal_frame, bg=self.colors["BG_LIGHT"],
                                 highlightbackground=self.colors["BORDER"],
                                 highlightthickness=1)
        terminal_body.pack(fill="both", expand=True)

        terminal_bg = "#0D1117" if self.is_dark_mode else "#F6F8FA"
        terminal_fg = "#C9D1D9" if self.is_dark_mode else "#24292F"
        terminal_cursor = "#58A6FF" if self.is_dark_mode else "#0969DA"

        self.txt_log = tk.Text(
            terminal_body, wrap="word",
            bg=terminal_bg, fg=terminal_fg,
            insertbackground=terminal_cursor,
            relief="flat", padx=15, pady=12,
            font=("Cascadia Code", 10)  # Reduzido de 13
        )
        self.txt_log.pack(side="left", fill="both", expand=True)

        scrollbar = ttk.Scrollbar(terminal_body, orient="vertical",
                                  command=self.txt_log.yview)
        scrollbar.pack(side="right", fill="y")
        self.txt_log.configure(yscrollcommand=scrollbar.set)

    # ---------- Execução ----------

    def _animate_progress(self):
        if hasattr(self, "progress_value"):
            width = self.progress_canvas.winfo_width()
            self.progress_canvas.coords(
                self.progress_bar,
                0, 0,
                width * self.progress_value, 10
            )
        self.after(50, self._animate_progress)

    def _on_proc_start(self, proc):
        self.current_proc = proc
        self.btn_stop.configure(state="normal")

    def _start_task(self, item):
        if self.current_worker and self.current_worker.is_alive():
            messagebox.showinfo(
                "⚠️ Execução em andamento",
                f"Aguarde terminar:\n{self.running_task_name}"
            )
            return

        self.running_task_name = item["label"]
        self.progress_value = 0.0
        self._set_status(f"🚀 Executando: {item['label']}", "info")
        self._disable_all(True)
        self.btn_stop.configure(state="disabled")
        self.current_proc = None

        self.current_worker = threading.Thread(
            target=self._worker_run, args=(item,), daemon=True
        )
        self.current_worker.start()

    def _worker_run(self, item):
        try:
            for v in (0.05, 0.18, 0.35):
                self.progress_value = v
                time.sleep(0.12)

            kind = item["kind"]
            path = os.path.join(self.base_dir, item["path"])
            
            if not os.path.exists(path):
                error_msg = f"❌ ARQUIVO NÃO ENCONTRADO: {path}"
                print(f"\n{'='*70}")
                print(error_msg)
                print(f"{'='*70}\n")
                self._set_status(error_msg, "error")
                return

            print("\n" + "="*70)
            print(f"🚀 Iniciando: {item['label']}")
            print(f"📂 Caminho: {path}")
            print("="*70 + "\n")

            if kind == "ipynb":
                code = run_notebook(path, on_start=self._on_proc_start)
            else:
                code = run_python(path, on_start=self._on_proc_start)

            self.progress_value = 1

            if code == 0:
                self._set_status(f"✅ {item['label']} concluído!", "success")
            else:
                self._set_status(f"❌ {item['label']} código {code}", "error")

            print("\n" + "="*70)
            print(f"✅ Fim: {item['label']} (exitcode={code})")
            print("="*70 + "\n")

        except FileNotFoundError as e:
            error_msg = f"❌ Arquivo não encontrado:\n{e}"
            self._set_status(error_msg, "error")
            print(f"\n{error_msg}\n")

        except Exception as e:
            error_msg = f"❌ Erro inesperado:\n{e}"
            self._set_status(error_msg, "error")
            print(f"\n{error_msg}\n")

        finally:
            self._disable_all(False)
            self.btn_stop.configure(state="disabled")
            self.current_proc = None
            self.current_worker = None
            self.running_task_name = None

    def _stop_task(self):
        if not self.current_worker:
            self._set_status("ℹ️ Nenhuma automação em execução.", "info")
            return

        if self.current_proc is None:
            self._set_status("ℹ️ Processo ainda iniciando...", "info")
            return

        self._set_status("🛑 Encerrando automação...", "error")
        _terminate_proc_tree(self.current_proc)
        print("\n[STOP] Processo encerrado manualmente.\n")

    def _disable_all(self, disable=True):
        state = "disabled" if disable else "normal"

        preserve = {
            self.btn_stop, self.btn_pbi, self.btn_theme, self.btn_sp
        }

        def apply(widget):
            if isinstance(widget, (tk.Button, ttk.Button)):
                if widget not in preserve:
                    try:
                        widget.configure(state=state)
                    except:
                        pass
            for c in widget.winfo_children():
                apply(c)

        apply(self)

    def _set_status(self, text, kind):
        colors = {
            "success": self.colors["ACCENT"],
            "error": self.colors["ERROR"],
            "info": self.colors["PRIMARY"],
            "warning": "#F59E0B"
        }
        self.status_label.configure(fg=colors.get(kind, self.colors["TEXT"]))
        self.status_var.set(text)

    def _clear_log(self):
        self.txt_log.configure(state="normal")
        self.txt_log.delete("1.0", "end")
        self.txt_log.configure(state="disabled")

    def _copy_log(self):
        txt = self.txt_log.get("1.0", "end-1c")
        self.clipboard_clear()
        self.clipboard_append(txt)
        messagebox.showinfo("📋", "Log copiado!")

    def _tick_clock(self):
        self.clock_var.set(datetime.now().strftime("%d/%m/%Y %H:%M:%S"))
        self.after(1000, self._tick_clock)

    def _toggle_theme(self):
        self.is_dark_mode = not self.is_dark_mode
        self.colors = apply_premium_theme(self, DARK_THEME if self.is_dark_mode else LIGHT_THEME)
        
        self.btn_theme.configure(
            text="🌙" if not self.is_dark_mode else "☀️"
        )
        
        for widget in self.winfo_children():
            widget.destroy()
        
        self._build_header()
        self._build_main_content()
        
        self._logger = PremiumLogger(self.txt_log)
        sys.stdout = self._logger
        sys.stderr = self._logger

    def _on_close(self):
        if self.current_worker and self.current_worker.is_alive():
            if not messagebox.askyesno("⚠️", "Encerrar automação em execução?"):
                return
            self._stop_task()
            self.after(600, self.destroy)
        else:
            self.destroy()

# =========================
# Main
# =========================

if __name__ == "__main__":
    if not os.path.exists(BASE_DIR):
        print(f"❌ Diretório base não encontrado: {BASE_DIR}")
        print(f"⚠️  Verifique se o caminho está correto.")
        print(f"📂 O diretório atual é: {os.getcwd()}")
        
        choice = input("Deseja usar o diretório atual como base? (s/n): ")
        if choice.lower() == 's':
            BASE_DIR = os.getcwd()
            print(f"✅ Usando diretório atual: {BASE_DIR}")
        else:
            input("Pressione Enter para sair...")
            sys.exit(1)
    
    app = GuardioesIAPremium(BASE_DIR, ITEMS)
    app.mainloop()

In [1]:
# -*- coding: utf-8 -*-
"""
Guardiões da IA — Central de Automação PREMIUM EDITION
VERSÃO COM EXECUÇÃO RPA SAFE (SAP / Excel / GUI)

Autor: Tiago Felix
"""

import os
import sys
import json
import queue
import threading
import subprocess
import time
import tempfile
import platform
from pathlib import Path
from datetime import datetime
import tkinter as tk
from tkinter import ttk, messagebox
import webbrowser

# =========================
# CONFIGURAÇÃO BÁSICA
# =========================

BASE_DIR = r"C:\Users\126815"

ITEMS = [
    {"label": "Gestor NFS ULTRA", "path": "RPA_GESTOR_NFS_ULTRA.ipynb", "kind": "ipynb", "icon": "📊", "color": "#3B82F6"},
    {"label": "Leitor Lumicenter", "path": "RPA_LEITOR_LUMICEN.ipynb", "kind": "ipynb", "icon": "💡", "color": "#F59E0B"},
    {"label": "Leitor Persun", "path": "RPA_LEITOR_PERSUN.ipynb", "kind": "ipynb", "icon": "📖", "color": "#8B5CF6"},
    {"label": "Tickets (E-mail)", "path": "RPA_TICKETS_E-MAIL.ipynb", "kind": "ipynb", "icon": "🎟️", "color": "#EC4899"},
    {"label": "RC SAP", "path": "RPA_RC_SAP.ipynb", "kind": "ipynb", "icon": "🔐", "color": "#06B6D4"},
    {"label": "RPA Bases RAW (ERP)", "path": "RPA_bases_raw.ipynb", "kind": "ipynb", "icon": "🗄️", "color": "#10B981"},
    {"label": "Pipeline Financeiro Completo", "path": "pipeline_financeiro_completo.ipynb", "kind": "ipynb", "icon": "📈", "color": "#22C55E"},
]

# =========================
# EXECUÇÃO DE NOTEBOOK COMO SCRIPT (RPA SAFE)
# =========================

def run_notebook_as_script(path_ipynb, on_start=None):
    path_ipynb = os.path.abspath(path_ipynb)

    if not os.path.exists(path_ipynb):
        raise FileNotFoundError(f"Notebook não encontrado: {path_ipynb}")

    nb_dir = os.path.dirname(path_ipynb)

    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"

    with tempfile.TemporaryDirectory() as tmpdir:
        subprocess.run(
            ["jupyter", "nbconvert", "--to", "script", path_ipynb, "--output-dir", tmpdir],
            check=True,
            cwd=nb_dir
        )

        py_files = list(Path(tmpdir).glob("*.py"))
        if not py_files:
            raise RuntimeError("Falha ao converter notebook")

        script_py = str(py_files[0])

        proc = subprocess.Popen(
            [sys.executable, script_py],
            cwd=nb_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=env
        )

        if on_start:
            on_start(proc)

        for line in proc.stdout:
            print(line, end="")

        return proc.wait()

# =========================
# A PARTIR DAQUI: SEU VISUAL ORIGINAL (INALTERADO)
# =========================

class GuardioesIAPremium(tk.Tk):
    def __init__(self, base_dir: str, items: list):
        super().__init__()
        self.title("🛡️ Guardiões da IA - Premium Edition")
        self.geometry("1250x750")
        self.minsize(1100, 650)

        self.base_dir = base_dir
        self.items = items
        self.current_worker = None
        self.current_proc = None

        self._build_ui()

    def _build_ui(self):
        self.configure(bg="#0A0E1A")

        main = tk.Frame(self, bg="#0A0E1A")
        main.pack(fill="both", expand=True, padx=20, pady=20)

        left = tk.Frame(main, bg="#0A0E1A", width=420)
        left.pack(side="left", fill="y")

        right = tk.Frame(main, bg="#020617")
        right.pack(side="right", fill="both", expand=True, padx=(20, 0))

        tk.Label(
            left, text="⚙️ Automatizações",
            bg="#0A0E1A", fg="#F1F5F9",
            font=("Segoe UI Semibold", 14)
        ).pack(anchor="w", pady=(0, 10))

        for item in self.items:
            btn = tk.Button(
                left,
                text=f"{item['icon']}  {item['label']}",
                bg="#1E293B", fg="#F1F5F9",
                relief="flat", padx=14, pady=10, anchor="w",
                command=lambda i=item: self._start_task(i)
            )
            btn.pack(fill="x", pady=6)

        tk.Label(
            right, text="💻 Terminal",
            bg="#020617", fg="#F1F5F9",
            font=("Segoe UI Semibold", 13)
        ).pack(anchor="w", padx=12, pady=(12, 6))

        self.txt_log = tk.Text(
            right,
            bg="#020617",
            fg="#E5E7EB",
            font=("Cascadia Code", 10),
            relief="flat"
        )
        self.txt_log.pack(fill="both", expand=True, padx=12, pady=6)
        self.txt_log.configure(state="disabled")

    def _start_task(self, item):
        if self.current_worker and self.current_worker.is_alive():
            messagebox.showinfo("⚠️ Execução em andamento", "Aguarde a automação atual terminar.")
            return

        path = os.path.join(self.base_dir, item["path"])
        self._log(f"\n🚀 Iniciando: {item['label']}\n📂 {path}\n")

        self.current_worker = threading.Thread(
            target=self._run_task, args=(item,), daemon=True
        )
        self.current_worker.start()

    def _run_task(self, item):
        try:
            code = run_notebook_as_script(os.path.join(self.base_dir, item["path"]))
            self._log(f"\n✅ Finalizado com sucesso (exit code {code})\n")
        except Exception as e:
            self._log(f"\n❌ Erro na execução:\n{e}\n")

    def _log(self, msg: str):
        self.txt_log.configure(state="normal")
        self.txt_log.insert("end", msg)
        self.txt_log.see("end")
        self.txt_log.configure(state="disabled")

# =========================
# MAIN
# =========================

if __name__ == "__main__":
    if not os.path.exists(BASE_DIR):
        messagebox.showerror("Erro", f"Diretório base não encontrado:\n{BASE_DIR}")
        sys.exit(1)

    app = GuardioesIAPremium(BASE_DIR, ITEMS)
    app.mainloop()

In [3]:
# -*- coding: utf-8 -*-
"""
Guardiões da IA — Central de Automação PREMIUM EDITION v2
Autor: Tiago Felix | Revisão técnica: 2026-04

Correções aplicadas:
  - Toggle de tema preserva o log (não destrói/recria widgets)
  - sys.stdout/stderr restaurados corretamente ao fechar
  - _popen_stream sempre recebe lista (nunca string)
  - _disable_all não percorre árvore recursiva desnecessariamente
  - progress_value inicializado no __init__
  - Arquivo _executado.ipynb removido após execução
  - on_start callback protegido contra race condition
  - Encoding UTF-8 forçado em todos os subprocessos Windows
  - Logger thread-safe via after() do tkinter
"""

import os
import sys
import json
import threading
import subprocess
import time
import tempfile
from pathlib import Path
from datetime import datetime
import platform
import tkinter as tk
from tkinter import ttk, messagebox
import webbrowser

# =========================
# CONFIGURAÇÃO
# =========================

BASE_DIR = r"C:\Users\126815"

ITEMS = [
    {"label": "Gestor NFS ULTRA",       "path": "RPA_GESTOR_NFS_ULTRA.ipynb",          "kind": "ipynb", "icon": "📊", "color": "#3B82F6"},
    {"label": "Leitor Lumicenter",       "path": "RPA_LEITOR_LUMICEN.ipynb",            "kind": "ipynb", "icon": "💡", "color": "#F59E0B"},
    {"label": "Leitor Persun",           "path": "RPA_LEITOR_PERSUN.ipynb",             "kind": "ipynb", "icon": "📖", "color": "#8B5CF6"},
    {"label": "N1 Zendesk",             "path": "RPA_N1_ZENDESK.ipynb",               "kind": "ipynb", "icon": "🎫", "color": "#10B981"},
    {"label": "Pedidos Coord (E-mail)", "path": "RPA_PEDIDOS_Coord_E-MAIL-.ipynb",    "kind": "ipynb", "icon": "📧", "color": "#EF4444"},
    {"label": "RC SAP",                 "path": "RPA_RC_SAP.ipynb",                   "kind": "ipynb", "icon": "🔐", "color": "#06B6D4"},
    {"label": "Renomear HTB",           "path": "RPA_RENOMEAR_HTB.ipynb",             "kind": "ipynb", "icon": "📝", "color": "#F97316"},
    {"label": "Tickets (E-mail)",       "path": "RPA_TICKETS_E-MAIL.ipynb",           "kind": "ipynb", "icon": "🎟️", "color": "#EC4899"},
]

PBI_MENU_URL = (
    r"https://app.powerbi.com/groups/603030a4-7459-45f3-ad62-07560039fba9"
    r"/reports/eb396a94-1c03-4751-b1e0-218f373e6ba1/e4668548a078e9dc9d0e?experience=power-bi"
)

SHAREPOINT_ENG_URL = (
    r"https://paguemenos-my.sharepoint.com/shared?"
    r"id=%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
    r"&listurl=https%3A%2F%2Fpaguemenos%2Esharepoint%2Ecom%2Fsites%2FENGENHARIA%2DOBRAS%2FDocumentos%20Compartilhados"
)

# =========================
# TEMAS
# =========================

DARK_THEME = {
    "BG":           "#0A0E1A",
    "BG_LIGHT":     "#131827",
    "SURFACE":      "#1E293B",
    "BORDER":       "#334155",
    "PRIMARY":      "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT":       "#10B981",
    "ERROR":        "#EF4444",
    "TEXT":         "#F1F5F9",
    "TEXT_MUTED":   "#94A3B8",
    "GLOW":         "#60A5FA",
    "TERMINAL_BG":  "#0D1117",
    "TERMINAL_FG":  "#C9D1D9",
    "TERMINAL_CUR": "#58A6FF",
}

LIGHT_THEME = {
    "BG":           "#F8FAFC",
    "BG_LIGHT":     "#FFFFFF",
    "SURFACE":      "#FFFFFF",
    "BORDER":       "#E2E8F0",
    "PRIMARY":      "#3B82F6",
    "PRIMARY_DARK": "#2563EB",
    "ACCENT":       "#10B981",
    "ERROR":        "#EF4444",
    "TEXT":         "#0F172A",
    "TEXT_MUTED":   "#64748B",
    "GLOW":         "#3B82F6",
    "TERMINAL_BG":  "#F6F8FA",
    "TERMINAL_FG":  "#24292F",
    "TERMINAL_CUR": "#0969DA",
}


def apply_theme_ttk(root: tk.Tk, theme_dict: dict):
    style = ttk.Style(root)
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass
    root.configure(bg=theme_dict["BG"])
    style.configure(".", background=theme_dict["BG"], foreground=theme_dict["TEXT"], font=("Segoe UI", 10))
    style.configure("TFrame", background=theme_dict["BG"])
    style.configure("Vertical.TScrollbar", background=theme_dict["SURFACE"], troughcolor=theme_dict["BG"])


# =========================
# LOGGER THREAD-SAFE
# =========================

class PremiumLogger:
    """
    Redireciona stdout/stderr para o widget Text.
    Usa root.after() para garantir que as escritas ocorram
    sempre na thread principal do Tkinter (thread-safe).
    """

    def __init__(self, root: tk.Tk, text_widget: tk.Text, max_lines: int = 20_000):
        self._root = root
        self._text = text_widget
        self._max_lines = max_lines
        self._lock = threading.Lock()
        self._setup_tags()

    def _setup_tags(self):
        self._text.tag_config("success", foreground="#10B981")
        self._text.tag_config("error",   foreground="#EF4444")
        self._text.tag_config("warning", foreground="#F59E0B")
        self._text.tag_config("info",    foreground="#3B82F6")

    def _detect_tag(self, msg: str):
        low = msg.lower()
        if "sucesso" in low or "✅" in msg:
            return "success"
        if "erro" in low or "❌" in msg or "error" in low or "traceback" in low:
            return "error"
        if "aviso" in low or "⚠️" in msg or "warning" in low:
            return "warning"
        if "[run]" in msg or "[kernels]" in msg or "🚀" in msg:
            return "info"
        return None

    def _do_write(self, msg: str):
        """Executado SEMPRE na thread principal via after()."""
        try:
            self._text.configure(state="normal")
            tag = self._detect_tag(msg)
            if tag:
                self._text.insert("end", msg, tag)
            else:
                self._text.insert("end", msg)

            # Limita linhas para não vazar memória
            lines = int(self._text.index("end-1c").split(".")[0])
            if lines > self._max_lines:
                self._text.delete("1.0", f"{lines - self._max_lines}.0")

            self._text.see("end")
            self._text.configure(state="disabled")
        except Exception:
            pass  # Widget pode ter sido destruído

    def write(self, msg: str):
        if not msg:
            return
        # Agenda na thread principal — nunca bloqueia a thread worker
        try:
            self._root.after(0, self._do_write, msg)
        except Exception:
            pass

    def flush(self):
        pass  # Não é necessário — escrita é assíncrona via after()

    def fileno(self):
        raise io.UnsupportedOperation("fileno")


# =========================
# HELPERS DE EXECUÇÃO
# =========================

def _open_url(url: str):
    try:
        webbrowser.open(url, new=2)
    except Exception as e:
        print(f"[dashboard] Falha ao abrir URL: {e}\n")


def _pick_kernel(prefer: str = "python3") -> str | None:
    try:
        proc = subprocess.run(
            ["jupyter", "kernelspec", "list", "--json"],
            capture_output=True, text=True, encoding="utf-8", errors="replace",
            timeout=15
        )
        data = json.loads(proc.stdout)
        kernels = list(data.get("kernelspecs", {}).keys())
        print(f"[kernels] Disponíveis: {kernels}\n")
        return prefer if prefer in kernels else (kernels[0] if kernels else None)
    except Exception as e:
        print(f"[kernels] Erro ao listar kernels: {e}\n")
        return None


def _make_env() -> dict:
    """Ambiente com UTF-8 forçado — essencial no Windows."""
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    return env


def _terminate_proc_tree(proc: subprocess.Popen | None):
    if proc is None:
        return
    try:
        if platform.system() == "Windows":
            subprocess.run(
                ["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                capture_output=True, timeout=10
            )
        else:
            import signal
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        try:
            proc.terminate()
        except Exception:
            pass


def _stream_proc(cmd: list[str], env: dict | None = None, cwd: str | None = None,
                 on_start=None) -> int:
    """
    Executa cmd como subprocesso, transmite stdout linha a linha para sys.stdout.
    on_start(proc) é chamado logo após o Popen — use para capturar a referência ao processo.
    Retorna o exit code.
    """
    print(f"[run] {' '.join(str(c) for c in cmd)}\n")

    creation_flags = subprocess.CREATE_NO_WINDOW if platform.system() == "Windows" else 0

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env or _make_env(),
        cwd=cwd,
        creationflags=creation_flags,
    )

    if on_start:
        on_start(proc)

    for line in proc.stdout:
        print(line, end="")

    return proc.wait()


def run_notebook_as_script(path_ipynb: str, on_start=None) -> int:
    """Fallback: converte para .py e executa com Python."""
    path_ipynb = os.path.abspath(path_ipynb)
    if not os.path.exists(path_ipynb):
        raise FileNotFoundError(f"Notebook não encontrado: {path_ipynb}")

    nb_dir = os.path.dirname(path_ipynb)
    nb_name = os.path.splitext(os.path.basename(path_ipynb))[0]
    env = _make_env()

    with tempfile.TemporaryDirectory() as tmpdir:
        code = _stream_proc(
            ["jupyter", "nbconvert", "--to", "script", path_ipynb, "--output-dir", tmpdir],
            env=env, cwd=nb_dir
        )
        if code != 0:
            return code

        candidates = list(Path(tmpdir).glob("*.py"))
        if not candidates:
            print("[run] Nenhum .py gerado pelo nbconvert.\n")
            return 1

        out_py = str(candidates[0])
        return _stream_proc([sys.executable, out_py], env=env, cwd=nb_dir, on_start=on_start)


def run_notebook(path_ipynb: str, on_start=None) -> int:
    """
    Tenta executar o notebook em três métodos, do mais robusto ao fallback.
    Limpa o arquivo _executado.ipynb após execução.
    """
    path_ipynb = os.path.abspath(path_ipynb)
    if not os.path.exists(path_ipynb):
        raise FileNotFoundError(f"Notebook não encontrado: {path_ipynb}")

    nb_dir = os.path.dirname(path_ipynb)
    nb_stem = os.path.splitext(os.path.basename(path_ipynb))[0]
    out_base = nb_stem + "_executado"
    out_file = os.path.join(nb_dir, out_base + ".ipynb")
    env = _make_env()

    kernel = _pick_kernel()

    if kernel:
        # Método 1: nbconvert execute
        cmd1 = [
            "jupyter", "nbconvert", "--to", "notebook", "--execute",
            path_ipynb, "--output", out_base,
            "--ExecutePreprocessor.kernel_name", kernel,
            "--ExecutePreprocessor.timeout=0",
        ]
        print("[run] Método 1 (nbconvert execute)…\n")
        code = _stream_proc(cmd1, env=env, cwd=nb_dir, on_start=on_start)
        _cleanup(out_file)
        if code == 0:
            return 0

        # Método 2: jupyter run
        print("[run] Método 1 falhou. Tentando método 2 (jupyter run)…\n")
        cmd2 = ["jupyter", "run", "--kernel", kernel, path_ipynb]
        code = _stream_proc(cmd2, env=env, cwd=nb_dir, on_start=on_start)
        if code == 0:
            return 0

    # Método 3: converte para .py e executa
    print("[run] Tentando método 3 (execução como script Python)…\n")
    return run_notebook_as_script(path_ipynb, on_start=on_start)


def run_python(path_py: str, on_start=None) -> int:
    path_py = os.path.abspath(path_py)
    return _stream_proc([sys.executable, path_py], env=_make_env(),
                        cwd=os.path.dirname(path_py), on_start=on_start)


def _cleanup(path: str):
    """Remove arquivo gerado se existir."""
    try:
        if os.path.exists(path):
            os.remove(path)
    except Exception:
        pass


# =========================
# APP PRINCIPAL
# =========================

class GuardioesIAPremium(tk.Tk):

    def __init__(self, base_dir: str, items: list[dict]):
        super().__init__()

        self.title("🛡️ Guardiões da IA - Premium Edition")
        self.is_dark = True
        self.C = DARK_THEME.copy()  # paleta atual

        # Tamanho responsivo
        sw, sh = self.winfo_screenwidth(), self.winfo_screenheight()
        w = min(1400, int(sw * 0.90))
        h = min(850,  int(sh * 0.88))
        x = (sw - w) // 2
        y = max(0, (sh - h) // 2 - 20)
        self.geometry(f"{w}x{h}+{x}+{y}")
        self.minsize(1100, 650)

        apply_theme_ttk(self, self.C)

        # Fontes
        self.F = {
            "base":     ("Segoe UI", 10),
            "small":    ("Segoe UI", 9),
            "medium":   ("Segoe UI Semibold", 12),
            "big":      ("Segoe UI Semibold", 15),
            "icon":     ("Segoe UI Emoji", 28),
            "terminal": ("Cascadia Code", 10),
        }

        # Estado de execução
        self._worker: threading.Thread | None = None
        self._proc: subprocess.Popen | None = None
        self._proc_lock = threading.Lock()
        self._task_name: str | None = None
        self.progress_value: float = 0.0  # ← inicializado aqui

        # Referências persistentes para o logger (não destruídas no toggle de tema)
        self._log_lines: list[str] = []

        self.base_dir = base_dir
        self.items = items

        # Stdout/stderr originais — restaurados ao fechar
        self._orig_stdout = sys.stdout
        self._orig_stderr = sys.stderr

        self._build_ui()

        # Logger aponta para o widget criado em _build_ui
        self._logger = PremiumLogger(self, self.txt_log)
        sys.stdout = self._logger
        sys.stderr = self._logger

        # Atalhos
        self.bind_all("<Control-m>", lambda _: _open_url(PBI_MENU_URL))
        self.bind_all("<Control-e>", lambda _: _open_url(SHAREPOINT_ENG_URL))
        self.bind_all("<Control-t>", lambda _: self._toggle_theme())

        self._tick_clock()
        self._animate_progress()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ------------------------------------------------------------------
    # CONSTRUÇÃO DA INTERFACE
    # ------------------------------------------------------------------

    def _build_ui(self):
        """Constrói/reconstrói apenas os widgets que dependem do tema."""
        apply_theme_ttk(self, self.C)
        self._build_header()
        self._build_main()

    def _rebuild_theme_colors(self):
        """
        Atualiza apenas as cores dos widgets existentes — sem destruir nada.
        Chamado pelo toggle de tema.
        """
        C = self.C

        # Header
        self.header_glass.configure(bg=C["BG_LIGHT"], highlightbackground=C["BORDER"])
        self.title_lbl.configure(bg=C["BG_LIGHT"], fg=C["TEXT"])
        self.subtitle_lbl.configure(bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"])
        self.dev_frame.configure(bg=C["BG_LIGHT"])
        self.dev_lbl1.configure(bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"])
        self.dev_lbl2.configure(bg=C["BG_LIGHT"], fg=C["GLOW"])
        self.btn_theme.configure(bg=C["SURFACE"], fg=C["TEXT"])
        self.btn_pbi.configure(bg=C["PRIMARY"])
        self.btn_sp.configure(bg=C["SURFACE"], fg=C["TEXT"])
        self.clock_label.configure(bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"])

        # Main
        self.main.configure(bg=C["BG"])

        # Terminal
        terminal_bg = C["TERMINAL_BG"]
        terminal_fg = C["TERMINAL_FG"]
        self.txt_log.configure(bg=terminal_bg, fg=terminal_fg, insertbackground=C["TERMINAL_CUR"])
        self.terminal_header.configure(bg=C["SURFACE"], highlightbackground=C["BORDER"])
        self.terminal_title.configure(bg=C["SURFACE"], fg=C["TEXT"])
        self.terminal_btns.configure(bg=C["SURFACE"])
        self.btn_clear.configure(bg=C["BG_LIGHT"], fg=C["TEXT"])
        self.btn_copy.configure(bg=C["BG_LIGHT"], fg=C["TEXT"])
        self.terminal_body.configure(bg=C["BG_LIGHT"], highlightbackground=C["BORDER"])

        # Status
        self.status_card.configure(bg=C["SURFACE"], highlightbackground=C["BORDER"])
        self.progress_canvas.configure(bg=C["BG_LIGHT"])
        self.progress_canvas.itemconfig(self.progress_bar, fill=C["PRIMARY"])
        self.btn_stop.configure(bg=C["ERROR"])

        # Cards
        for card_info in self._card_refs:
            card, item = card_info["card"], card_info["item"]
            card.configure(bg=C["SURFACE"], highlightbackground=C["BORDER"])
            card_info["icon"].configure(bg=C["SURFACE"], fg=item["color"])
            card_info["txt_frame"].configure(bg=C["SURFACE"])
            card_info["title"].configure(bg=C["SURFACE"], fg=C["TEXT"])
            card_info["desc"].configure(bg=C["SURFACE"], fg=C["TEXT_MUTED"])
            card_info["arrow"].configure(bg=C["SURFACE"], fg=item["color"])

        self.configure(bg=C["BG"])

    # ------ Header ------

    def _build_header(self):
        if hasattr(self, "_header_frame"):
            self._header_frame.destroy()

        C = self.C
        frame = tk.Frame(self, bg=C["BG"], height=110)
        frame.pack(fill="x")
        frame.pack_propagate(False)
        self._header_frame = frame

        glass = tk.Frame(frame, bg=C["BG_LIGHT"],
                         highlightbackground=C["BORDER"], highlightthickness=1)
        glass.pack(fill="both", expand=True, padx=20, pady=12)
        self.header_glass = glass

        # Esquerda
        left = tk.Frame(glass, bg=C["BG_LIGHT"])
        left.pack(side="left", padx=15, pady=10)

        self.title_lbl = tk.Label(left, text="🛡️ Guardiões da IA",
                                   bg=C["BG_LIGHT"], fg=C["TEXT"],
                                   font=("Segoe UI Semibold", 16, "bold"))
        self.title_lbl.pack(anchor="w")

        self.subtitle_lbl = tk.Label(left, text="Central de Automação Premium",
                                      bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"],
                                      font=("Segoe UI", 10))
        self.subtitle_lbl.pack(anchor="w", pady=(4, 0))

        # Centro
        center = tk.Frame(glass, bg=C["BG_LIGHT"])
        center.pack(side="left", expand=True, fill="both")

        self.dev_frame = tk.Frame(center, bg=C["BG_LIGHT"])
        self.dev_frame.place(relx=0.5, rely=0.5, anchor="center")

        self.dev_lbl1 = tk.Label(self.dev_frame, text="🚀 Desenvolvido por:",
                                  bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"],
                                  font=("Segoe UI", 9, "bold"))
        self.dev_lbl1.pack()

        self.dev_lbl2 = tk.Label(self.dev_frame, text="TIAGO FELIX",
                                  bg=C["BG_LIGHT"], fg=C["GLOW"],
                                  font=("Segoe UI Semibold", 13, "bold"))
        self.dev_lbl2.pack(pady=(1, 0))

        # Direita
        right = tk.Frame(glass, bg=C["BG_LIGHT"])
        right.pack(side="right", padx=12, pady=10)

        btns = tk.Frame(right, bg=C["BG_LIGHT"])
        btns.pack(anchor="e", pady=(0, 6))

        self.btn_theme = tk.Button(btns, text="☀️", command=self._toggle_theme,
                                    bg=C["SURFACE"], fg=C["TEXT"],
                                    relief="flat", font=("Segoe UI", 14),
                                    padx=8, pady=4, cursor="hand2")
        self.btn_theme.pack(side="left", padx=(0, 6))

        self.btn_pbi = tk.Button(btns, text="📊 Power BI",
                                  command=lambda: _open_url(PBI_MENU_URL),
                                  bg=C["PRIMARY"], fg="white", relief="flat",
                                  font=self.F["small"], padx=12, pady=5, cursor="hand2")
        self.btn_pbi.pack(side="left", padx=(0, 6))

        self.btn_sp = tk.Button(btns, text="🏗️ SharePoint",
                                 command=lambda: _open_url(SHAREPOINT_ENG_URL),
                                 bg=C["SURFACE"], fg=C["TEXT"], relief="flat",
                                 font=self.F["small"], padx=12, pady=5, cursor="hand2")
        self.btn_sp.pack(side="left")

        self.clock_var = tk.StringVar()
        self.clock_label = tk.Label(right, textvariable=self.clock_var,
                                     bg=C["BG_LIGHT"], fg=C["TEXT_MUTED"],
                                     font=("Segoe UI", 9))
        self.clock_label.pack(anchor="e", pady=(6, 0))

    # ------ Main ------

    def _build_main(self):
        if hasattr(self, "main"):
            self.main.destroy()

        C = self.C
        self.main = tk.Frame(self, bg=C["BG"])
        self.main.pack(fill="both", expand=True, padx=20, pady=(8, 15))
        self.main.grid_columnconfigure(0, weight=2)
        self.main.grid_columnconfigure(1, weight=3)
        self.main.grid_rowconfigure(0, weight=1)

        self._card_refs = []
        self._build_cards_panel()
        self._build_terminal_panel()

    # ------ Painel de cards ------

    def _build_cards_panel(self):
        C = self.C
        left = tk.Frame(self.main, bg=C["BG"])
        left.grid(row=0, column=0, sticky="nsew", padx=(0, 15))

        tk.Label(left, text="⚙️ Automatizações",
                 bg=C["BG"], fg=C["TEXT"],
                 font=("Segoe UI Semibold", 13)).pack(anchor="w", pady=(0, 10))

        canvas_frame = tk.Frame(left, bg=C["BG"])
        canvas_frame.pack(fill="both", expand=True)

        canvas = tk.Canvas(canvas_frame, bg=C["BG"], highlightthickness=0)
        scroll = ttk.Scrollbar(canvas_frame, orient="vertical", command=canvas.yview)
        inner = tk.Frame(canvas, bg=C["BG"])

        inner.bind("<Configure>", lambda _: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=inner, anchor="nw")
        canvas.configure(yscrollcommand=scroll.set)

        canvas.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        # Scroll com mouse
        canvas.bind("<Enter>", lambda _: canvas.bind_all("<MouseWheel>",
            lambda e: canvas.yview_scroll(int(-e.delta / 120), "units")))
        canvas.bind("<Leave>", lambda _: canvas.unbind_all("<MouseWheel>"))

        for item in self.items:
            self._make_card(inner, item)

    def _make_card(self, parent, item):
        C = self.C

        card = tk.Frame(parent, bg=C["SURFACE"],
                        highlightbackground=C["BORDER"], highlightthickness=1,
                        cursor="hand2")
        card.pack(fill="x", pady=8, ipady=10, ipadx=12)

        def on_enter(_):
            card.configure(bg=C["BG_LIGHT"], highlightbackground=item["color"])
            for ref in (left, txt_frame):
                ref.configure(bg=C["BG_LIGHT"])
            for ref in (icon_lbl, title_lbl, desc_lbl, arrow_lbl):
                ref.configure(bg=C["BG_LIGHT"])

        def on_leave(_):
            card.configure(bg=C["SURFACE"], highlightbackground=C["BORDER"])
            for ref in (left, txt_frame):
                ref.configure(bg=C["SURFACE"])
            for ref in (icon_lbl, title_lbl, desc_lbl, arrow_lbl):
                ref.configure(bg=C["SURFACE"])

        card.bind("<Enter>", on_enter)
        card.bind("<Leave>", on_leave)
        card.bind("<Button-1>", lambda _: self._start_task(item))

        left = tk.Frame(card, bg=C["SURFACE"])
        left.pack(side="left", fill="both", expand=True)

        icon_lbl = tk.Label(left, text=item["icon"], bg=C["SURFACE"],
                             fg=item["color"], font=("Segoe UI Emoji", 28))
        icon_lbl.pack(side="left", padx=(6, 12))

        txt_frame = tk.Frame(left, bg=C["SURFACE"])
        txt_frame.pack(side="left", fill="both", expand=True)

        title_lbl = tk.Label(txt_frame, text=item["label"],
                              bg=C["SURFACE"], fg=C["TEXT"],
                              font=("Segoe UI Semibold", 11))
        title_lbl.pack(anchor="w", pady=(1, 0))

        desc_lbl = tk.Label(txt_frame, text=f"Executar {item['kind'].upper()}",
                             bg=C["SURFACE"], fg=C["TEXT_MUTED"],
                             font=("Segoe UI", 9))
        desc_lbl.pack(anchor="w")

        arrow_lbl = tk.Label(card, text="▶", bg=C["SURFACE"],
                              fg=item["color"], font=("Segoe UI", 16))
        arrow_lbl.pack(side="right", padx=10)

        # Bind clique em todos os filhos
        for w in (left, icon_lbl, txt_frame, title_lbl, desc_lbl, arrow_lbl):
            w.bind("<Button-1>", lambda _: self._start_task(item))
            w.bind("<Enter>", on_enter)
            w.bind("<Leave>", on_leave)

        self._card_refs.append({
            "card": card, "item": item,
            "icon": icon_lbl, "txt_frame": txt_frame,
            "title": title_lbl, "desc": desc_lbl, "arrow": arrow_lbl,
        })

    # ------ Painel terminal ------

    def _build_terminal_panel(self):
        C = self.C
        right = tk.Frame(self.main, bg=C["BG"])
        right.grid(row=0, column=1, sticky="nsew")

        # Barra de status
        self.status_card = tk.Frame(right, bg=C["SURFACE"],
                                     highlightbackground=C["BORDER"], highlightthickness=1)
        self.status_card.pack(fill="x", pady=(0, 12))

        progress_frame = tk.Frame(self.status_card, bg=C["SURFACE"])
        progress_frame.pack(fill="x", padx=15, pady=(12, 8))

        self.progress_canvas = tk.Canvas(progress_frame, height=10,
                                          bg=C["BG_LIGHT"], highlightthickness=0)
        self.progress_canvas.pack(fill="x", expand=True, side="left")
        self.progress_bar = self.progress_canvas.create_rectangle(
            0, 0, 0, 10, fill=C["PRIMARY"], outline=""
        )

        self.btn_stop = tk.Button(progress_frame, text="⏹", command=self._stop_task,
                                   bg=C["ERROR"], fg="white", relief="flat",
                                   font=("Segoe UI", 12), padx=12, pady=5,
                                   state="disabled", cursor="hand2")
        self.btn_stop.pack(side="right", padx=(10, 0))

        self.status_var = tk.StringVar(value="🎯 Pronto para iniciar")
        self.status_label = tk.Label(self.status_card, textvariable=self.status_var,
                                      bg=C["SURFACE"], fg=C["PRIMARY"],
                                      font=("Segoe UI Semibold", 11), anchor="w")
        self.status_label.pack(fill="x", padx=15, pady=(0, 12))

        # Terminal
        terminal_frame = tk.Frame(right, bg=C["BG"])
        terminal_frame.pack(fill="both", expand=True)

        self.terminal_header = tk.Frame(terminal_frame, bg=C["SURFACE"],
                                         highlightbackground=C["BORDER"], highlightthickness=1)
        self.terminal_header.pack(fill="x")

        self.terminal_title = tk.Label(self.terminal_header, text="💻 Terminal",
                                        bg=C["SURFACE"], fg=C["TEXT"],
                                        font=("Segoe UI Semibold", 12))
        self.terminal_title.pack(side="left", padx=15, pady=10)

        self.terminal_btns = tk.Frame(self.terminal_header, bg=C["SURFACE"])
        self.terminal_btns.pack(side="right", padx=15, pady=10)

        self.btn_clear = tk.Button(self.terminal_btns, text="🗑️", command=self._clear_log,
                                    bg=C["BG_LIGHT"], fg=C["TEXT"],
                                    relief="flat", font=("Segoe UI", 11),
                                    padx=10, pady=4, cursor="hand2")
        self.btn_clear.pack(side="left", padx=(0, 8))

        self.btn_copy = tk.Button(self.terminal_btns, text="📋", command=self._copy_log,
                                   bg=C["BG_LIGHT"], fg=C["TEXT"],
                                   relief="flat", font=("Segoe UI", 11),
                                   padx=10, pady=4, cursor="hand2")
        self.btn_copy.pack(side="left")

        self.terminal_body = tk.Frame(terminal_frame, bg=C["BG_LIGHT"],
                                       highlightbackground=C["BORDER"], highlightthickness=1)
        self.terminal_body.pack(fill="both", expand=True)

        self.txt_log = tk.Text(
            self.terminal_body,
            wrap="word",
            bg=C["TERMINAL_BG"], fg=C["TERMINAL_FG"],
            insertbackground=C["TERMINAL_CUR"],
            relief="flat", padx=15, pady=12,
            font=self.F["terminal"],
            state="disabled",
        )
        self.txt_log.pack(side="left", fill="both", expand=True)

        sb = ttk.Scrollbar(self.terminal_body, orient="vertical", command=self.txt_log.yview)
        sb.pack(side="right", fill="y")
        self.txt_log.configure(yscrollcommand=sb.set)

    # ------------------------------------------------------------------
    # EXECUÇÃO DE TAREFAS
    # ------------------------------------------------------------------

    def _on_proc_start(self, proc: subprocess.Popen):
        with self._proc_lock:
            self._proc = proc
        self.after(0, lambda: self.btn_stop.configure(state="normal"))

    def _start_task(self, item: dict):
        if self._worker and self._worker.is_alive():
            messagebox.showinfo("⚠️ Em andamento", f"Aguarde terminar:\n{self._task_name}")
            return

        self._task_name = item["label"]
        self.progress_value = 0.0
        self._set_status(f"🚀 Executando: {item['label']}", "info")
        self._set_cards_state("disabled")
        self.btn_stop.configure(state="disabled")

        self._worker = threading.Thread(target=self._worker_run, args=(item,), daemon=True)
        self._worker.start()

    def _worker_run(self, item: dict):
        try:
            for v in (0.05, 0.18, 0.35):
                self.progress_value = v
                time.sleep(0.1)

            path = os.path.join(self.base_dir, item["path"])

            if not os.path.exists(path):
                msg = f"❌ Arquivo não encontrado: {path}"
                print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
                self.after(0, self._set_status, msg, "error")
                return

            print(f"\n{'='*70}\n🚀 Iniciando: {item['label']}\n📂 Caminho: {path}\n{'='*70}\n")

            kind = item["kind"]
            if kind == "ipynb":
                code = run_notebook(path, on_start=self._on_proc_start)
            else:
                code = run_python(path, on_start=self._on_proc_start)

            self.progress_value = 1.0

            if code == 0:
                status_msg = f"✅ {item['label']} concluído com sucesso!"
                status_kind = "success"
            else:
                status_msg = f"❌ {item['label']} finalizou com código {code}"
                status_kind = "error"

            print(f"\n{'='*70}\n{status_msg} (exitcode={code})\n{'='*70}\n")
            self.after(0, self._set_status, status_msg, status_kind)

        except FileNotFoundError as e:
            msg = f"❌ Arquivo não encontrado:\n{e}"
            print(f"\n{msg}\n")
            self.after(0, self._set_status, msg, "error")

        except Exception as e:
            import traceback
            msg = f"❌ Erro inesperado: {e}"
            print(f"\n{msg}\n{traceback.format_exc()}\n")
            self.after(0, self._set_status, msg, "error")

        finally:
            self.after(0, self._task_cleanup)

    def _task_cleanup(self):
        """Restaura UI após tarefa — chamado na thread principal via after()."""
        self._set_cards_state("normal")
        self.btn_stop.configure(state="disabled")
        with self._proc_lock:
            self._proc = None
        self._worker = None
        self._task_name = None

    def _stop_task(self):
        if not self._worker or not self._worker.is_alive():
            self._set_status("ℹ️ Nenhuma automação em execução.", "info")
            return
        with self._proc_lock:
            proc = self._proc
        if proc is None:
            self._set_status("ℹ️ Processo ainda iniciando…", "info")
            return

        self._set_status("🛑 Encerrando automação…", "error")
        _terminate_proc_tree(proc)
        print("\n[STOP] Processo encerrado manualmente.\n")

    def _set_cards_state(self, state: str):
        """Habilita/desabilita apenas os cards — sem percorrer toda a árvore."""
        for ref in getattr(self, "_card_refs", []):
            try:
                ref["card"].configure(cursor="hand2" if state == "normal" else "")
            except Exception:
                pass

    # ------------------------------------------------------------------
    # UI HELPERS
    # ------------------------------------------------------------------

    def _set_status(self, text: str, kind: str):
        colors = {
            "success": self.C["ACCENT"],
            "error":   self.C["ERROR"],
            "info":    self.C["PRIMARY"],
            "warning": "#F59E0B",
        }
        self.status_label.configure(fg=colors.get(kind, self.C["TEXT"]))
        self.status_var.set(text)

    def _clear_log(self):
        self.txt_log.configure(state="normal")
        self.txt_log.delete("1.0", "end")
        self.txt_log.configure(state="disabled")

    def _copy_log(self):
        txt = self.txt_log.get("1.0", "end-1c")
        self.clipboard_clear()
        self.clipboard_append(txt)
        messagebox.showinfo("📋", "Log copiado para a área de transferência!")

    def _animate_progress(self):
        try:
            w = self.progress_canvas.winfo_width()
            self.progress_canvas.coords(self.progress_bar, 0, 0, w * self.progress_value, 10)
        except Exception:
            pass
        self.after(50, self._animate_progress)

    def _tick_clock(self):
        try:
            self.clock_var.set(datetime.now().strftime("%d/%m/%Y %H:%M:%S"))
        except Exception:
            pass
        self.after(1000, self._tick_clock)

    def _toggle_theme(self):
        """
        Alterna o tema SEM destruir os widgets.
        Apenas repinta as cores dos widgets existentes.
        """
        self.is_dark = not self.is_dark
        self.C = DARK_THEME.copy() if self.is_dark else LIGHT_THEME.copy()
        apply_theme_ttk(self, self.C)

        self.btn_theme.configure(text="☀️" if self.is_dark else "🌙")

        # Reconstrói header (simples, sem log) e repinta o restante
        self._build_header()
        self._rebuild_theme_colors()

        # Logger continua apontando para o mesmo txt_log — log preservado
        if hasattr(self, "_logger"):
            self._logger._setup_tags()

    def _on_close(self):
        if self._worker and self._worker.is_alive():
            if not messagebox.askyesno("⚠️", "Encerrar automação em execução e fechar?"):
                return
            self._stop_task()
            self.after(800, self._shutdown)
        else:
            self._shutdown()

    def _shutdown(self):
        sys.stdout = self._orig_stdout
        sys.stderr = self._orig_stderr
        self.destroy()


# =========================
# ENTRY POINT
# =========================

if __name__ == "__main__":
    import io  # necessário para PremiumLogger.fileno()

    if not os.path.exists(BASE_DIR):
        print(f"❌ Diretório base não encontrado: {BASE_DIR}")
        print(f"📂 Diretório atual: {os.getcwd()}")
        choice = input("Usar o diretório atual como base? (s/n): ").strip().lower()
        if choice == "s":
            BASE_DIR = os.getcwd()
            print(f"✅ Usando: {BASE_DIR}")
        else:
            input("Pressione Enter para sair…")
            sys.exit(1)

    app = GuardioesIAPremium(BASE_DIR, ITEMS)
    app.mainloop()